# Math model for monoplane wing (no flaps/slats)

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

def read_xfoil_polar(filename):
    with open(filename) as f:
        lines = f.readlines()
    header_idx = None
    for i, line in enumerate(lines):
        if line.strip().startswith('Alpha'):
            header_idx = i
            break
    if header_idx is None:
        raise ValueError('Header not found (no "Alpha" line)')
    from io import StringIO
    data_str = '\n'.join(lines[header_idx:])
    df = pd.read_csv(StringIO(data_str))
    alpha_arr, cl_arr, cd_arr = df['Alpha'].values, df['Cl'].values, df['Cd'].values
    return alpha_arr, cl_arr, cd_arr

# --- User Inputs ---
inputs = {
    "root_polar": "/content/xf-s1210-il-500000.csv",
    "tip_polar":  "/content/xf-s1210-il-500000.csv",
    "mean_chord": 0.073,              # Fixed mean chord (m)
    "root_twist": 0.0,                # Root twist (deg)
    "tip_twist": np.linspace(0.0, -5 , 10),  # Tip twist range (deg)
    "n_sections": 30,
    "area_range": np.linspace(0.05, 0.1, 10),
    "tip_ratio_range": np.linspace(0.4, 0.6, 5)  # tip/root chord ratio range
}
results = []

# -- Airfoil polars
alpha_root, cl_root, cd_root = read_xfoil_polar(inputs["root_polar"])
alpha_tip, cl_tip, cd_tip = read_xfoil_polar(inputs["tip_polar"])
cl_interp_root = interp1d(alpha_root, cl_root, bounds_error=False, fill_value='extrapolate')
cd_interp_root = interp1d(alpha_root, cd_root, bounds_error=False, fill_value='extrapolate')
cl_interp_tip  = interp1d(alpha_tip,  cl_tip,  bounds_error=False, fill_value='extrapolate')
cd_interp_tip  = interp1d(alpha_tip,  cd_tip,  bounds_error=False, fill_value='extrapolate')
stall_angle_root = alpha_root[np.argmax(cl_root)]
stall_angle_tip  = alpha_tip[np.argmax(cl_tip)]
max_aoa = min(stall_angle_root, stall_angle_tip)
aoa_list = np.linspace(0, max_aoa, 20)

# --- Taper Functions ---
def chord_linear(blend, root_chord, tip_chord):
    return root_chord + (tip_chord - root_chord) * blend

def chord_piecewise(blend, root_chord, tip_chord, p):
    chord = np.full_like(blend, root_chord)
    mask = blend > p
    chord[mask] = root_chord + (tip_chord - root_chord) * (blend[mask] - p) / (1 - p)
    return chord

def chord_quadratic(blend, root_chord, tip_chord):
    return root_chord * (1 - blend**2) + tip_chord * blend**2

def chord_elliptic(blend, root_chord, tip_chord):
    chord = root_chord * np.sqrt(1 - blend**2) + tip_chord * blend
    return chord

# --- Utility Functions ---
def compute_area(chord_distribution, y):
    # Integrate chord along half-span and double for both wings
    area = 2 * np.trapz(chord_distribution, x=y)
    return area

def span_for_area(target_area, chord_func, root_chord, tip_chord, n_sections):
    # Find span such that integrated area matches target_area using bisection
    span_min, span_max = 0.1, 10.0
    for _ in range(20):
        span_guess = 0.5 * (span_min + span_max)
        y = np.linspace(0, span_guess/2, n_sections)
        blend = y / (span_guess/2)
        chord = chord_func(blend, root_chord, tip_chord)
        area_calc = compute_area(chord, y)
        if area_calc < target_area:
            span_min = span_guess
        else:
            span_max = span_guess
    return span_guess

# --- Blend-based wing sectional CL/CD ---
def spanwise_CL_CD(alpha, blend):
    CL_sec = (1 - blend) * cl_interp_root(alpha) + blend * cl_interp_tip(alpha)
    CD_sec = (1 - blend) * cd_interp_root(alpha) + blend * cd_interp_tip(alpha)
    return CL_sec, CD_sec

# --- Main parametric sweep ---
for area in inputs["area_range"]:
    mean_chord = inputs["mean_chord"]
    for tip_ratio in inputs["tip_ratio_range"]:
        root_chord = 2 * mean_chord / (1 + tip_ratio)
        tip_chord = tip_ratio * root_chord

        for tip_twist in inputs["tip_twist"]:
            n_sections = inputs["n_sections"]

            for taper_method in ["linear", "piecewise", "quadratic", "elliptic"]:
                if taper_method == "linear":
                    chord_func = chord_linear
                elif taper_method == "piecewise":
                    chord_func = None  # Will handle separately
                elif taper_method == "quadratic":
                    chord_func = chord_quadratic
                elif taper_method == "elliptic":
                    chord_func = chord_elliptic

                if taper_method == "piecewise":
                    for p in np.linspace(0.1, 1, 10):
                        def chord_p(blend, rc, tc):
                            return chord_piecewise(blend, rc, tc, p)
                        span = span_for_area(area, chord_p, root_chord, tip_chord, n_sections)
                        y = np.linspace(0, span/2, n_sections)
                        blend = y / (span/2)
                        chords = chord_p(blend, root_chord, tip_chord)
                        S = compute_area(chords, y)
                        AR = span / (S / span) if span > 0 else 0

                        for aoa in aoa_list:
                            twist_distribution = np.linspace(inputs["root_twist"], tip_twist, n_sections)
                            alpha_eff = aoa + twist_distribution
                            CL_section, CD_section = np.zeros(n_sections), np.zeros(n_sections)
                            for i in range(n_sections):
                                CL, CD = spanwise_CL_CD(alpha=alpha_eff[i], blend=blend[i])
                                CL_section[i], CD_section[i] = CL, CD
                            CL_wing = (2/S) * np.trapz(CL_section * chords, x=y)
                            CD_wing = (2/S) * np.trapz(CD_section * chords, x=y)
                            alpha_i_rad = CL_wing / (np.pi * AR)
                            alpha_i_deg = np.degrees(alpha_i_rad)
                            alpha_eff_corr = aoa + twist_distribution - alpha_i_deg
                            CL_section_corr, CD_section_corr = np.zeros(n_sections), np.zeros(n_sections)
                            for i in range(n_sections):
                                CL, CD = spanwise_CL_CD(alpha=alpha_eff_corr[i], blend=blend[i])
                                CL_section_corr[i], CD_section_corr[i] = CL, CD
                            CL_wing_corr = (2/S) * np.trapz(CL_section_corr * chords, x=y)
                            CD_wing_corr = (2/S) * np.trapz(CD_section_corr * chords, x=y)
                            CDi_integral = (2/S) * np.trapz((CL_section_corr**2) * chords, x=y)
                            CDi_actual = CDi_integral / (np.pi * AR) if AR > 0 else 0
                            results.append({
                                "AR": AR,
                                "span": span,
                                "area": S,
                                "root_chord": root_chord,
                                "tip_chord": tip_chord,
                                "mean_chord": mean_chord,
                                "twist": tip_twist,
                                "AoA": aoa,
                                "taper": taper_method,
                                "p": p,
                                "CL": CL_wing_corr,
                                "CDi": CDi_actual,
                                "CD": CD_wing_corr + CDi_actual,
                            })
                else:
                    span = span_for_area(area, chord_func, root_chord, tip_chord, n_sections)
                    y = np.linspace(0, span/2, n_sections)
                    blend = y / (span/2)
                    chords = chord_func(blend, root_chord, tip_chord)
                    S = compute_area(chords, y)
                    AR = span / (S / span) if span > 0 else 0

                    for aoa in aoa_list:
                        twist_distribution = np.linspace(inputs["root_twist"], tip_twist, n_sections)
                        alpha_eff = aoa + twist_distribution
                        CL_section, CD_section = np.zeros(n_sections), np.zeros(n_sections)
                        for i in range(n_sections):
                            CL, CD = spanwise_CL_CD(alpha=alpha_eff[i], blend=blend[i])
                            CL_section[i], CD_section[i] = CL, CD
                        CL_wing = (2/S) * np.trapz(CL_section * chords, x=y)
                        CD_wing = (2/S) * np.trapz(CD_section * chords, x=y)
                        alpha_i_rad = CL_wing / (np.pi * AR)
                        alpha_i_deg = np.degrees(alpha_i_rad)
                        alpha_eff_corr = aoa + twist_distribution - alpha_i_deg
                        CL_section_corr, CD_section_corr = np.zeros(n_sections), np.zeros(n_sections)
                        for i in range(n_sections):
                            CL, CD = spanwise_CL_CD(alpha=alpha_eff_corr[i], blend=blend[i])
                            CL_section_corr[i], CD_section_corr[i] = CL, CD
                        CL_wing_corr = (2/S) * np.trapz(CL_section_corr * chords, x=y)
                        CD_wing_corr = (2/S) * np.trapz(CD_section_corr * chords, x=y)
                        CDi_integral = (2/S) * np.trapz((CL_section_corr**2) * chords, x=y)
                        CDi_actual = CDi_integral / (np.pi * AR) if AR > 0 else 0
                        results.append({
                            "AR": AR,
                            "span": span,
                            "area": S,
                            "root_chord": root_chord,
                            "tip_chord": tip_chord,
                            "mean_chord": mean_chord,
                            "twist": tip_twist,
                            "AoA": aoa,
                            "taper": taper_method,
                            "CL": CL_wing_corr,
                            "CDi": CDi_actual,
                            "CD": CD_wing_corr + CDi_actual,
                        })

# --- Display Table ---
df = pd.DataFrame(results)

# Analyze for best CL
max_idx = df['CL'].idxmax()
best_params = df.iloc[max_idx]
print("Best CL configuration:", best_params)

# To save results:
df.to_csv('wing_parametric_results_5e5.csv')


# Model with flaps

In [ ]:
import numpy as np

def calculate_delta_cl_3d_wing(
    aspect_ratio,       # A: Wing aspect ratio (b^2/S)
    chord,              # c: Mean aerodynamic chord (m)
    span,               # b: Wing span (m)
    taper_ratio,        # lambda: Tip chord / root chord (unitless)
    sweep_half_chord,   # Λ_c/2: Sweep angle at half chord (deg)
    angle_of_attack,    # α: Angle of attack (deg)
    lift_curve_slope_2d,# a0: 2D lift curve slope of airfoil (per radian)
    flap_chord_ratio,   # c_f/c: flap chord ratio (unitless)
    flap_span_ratio,    # b_f/b: flap span ratio (unitless)
    flap_deflection_deg,# δ: Flap deflection angle (deg)
    lift_curve_slope_flap_2d # a_cl_flap: 2D flap lift slope (per rad or deg)
):

    sweep_rad = np.radians(sweep_half_chord)
    alpha_rad = np.radians(angle_of_attack)
    flap_deflection_rad = np.radians(flap_deflection_deg)

    a0 = lift_curve_slope_2d

    # Corrected 3D lift curve slope Cla from NACA TN 3911 formula considering sweep and aspect ratio
    denom = 2 + np.sqrt(4 + aspect_ratio**2 * (1 + np.tan(sweep_rad)**2))
    Cla = (2 * np.pi * aspect_ratio * np.cos(sweep_rad)) / denom

    # Aerodynamic induction factor Ka normalized with 2D thin airfoil theory slope
    Ka = Cla / (2 * np.pi)

    # Flap chord effectiveness factor Kc (empirical)
    Kc = 1 + 0.5 * flap_chord_ratio * aspect_ratio / (aspect_ratio + 3)

    # Flap span effectiveness factor Kb (depends on flap span and taper)
    Kb = flap_span_ratio**2 * (1 + taper_ratio)

    # Flap lift slope must be converted to per radian if given per degree
    if lift_curve_slope_flap_2d < 1.0:  # assume per degree if less than 1
        a_flap_effective = np.radians(lift_curve_slope_flap_2d)
    else:
        a_flap_effective = lift_curve_slope_flap_2d

    # Section lift increment due to flap deflection
    delta_cla_section = a_flap_effective * flap_deflection_rad

    # 3D flap lift effectiveness factor
    Kf = Ka * Kb * Kc

    # 3D lift increment due to flap
    delta_cl = delta_cla_section * Kf

    # Baseline 3D wing lift coefficient without flap at given angle of attack
    baseline_CL = Cla * alpha_rad

    # Total lift coefficient including flap lift increment
    CL_total = baseline_CL + delta_cl

    return {
        "CL_new": CL_total,
        "Cla (3D lift-curve slope per rad)": Cla,
        "Ka (Aerodynamic induction factor)": Ka,
        "Kb (Flap span effectiveness factor)": Kb,
        "Kc (Flap chord effectiveness factor)": Kc,
        "Kf (3D flap effectiveness factor)": Kf,
        "Delta_cla_section (Section lift increment)": delta_cla_section,
        "baseline_CL": baseline_CL,
    }


# ------ Usage Example ------

# Typical parameters (example)
inputs = {
    "aspect_ratio": 10.9898,      # Wing aspect ratio (b^2/S)
    "chord": 0.0857,               # Mean aerodynamic chord (m)
    "span": 0.09883,                 # Wing span (m)
    "taper_ratio": 1,             # Tip chord / root chord
    "sweep_half_chord": 0,        # Sweep angle at half chord (deg)
    "angle_of_attack": 10,     # Angle of attack (deg)
    "lift_curve_slope_2d": 6.28, # 2D lift curve slope (per radian), approx. 2*pi
    "flap_chord_ratio": 0.2,      # Flap chord / wing chord
    "flap_span_ratio": 0.6,       # Flap span / wing span -- 30% is left for ailerons and 10% from root chord is left
    "flap_deflection_deg": 20,    # Flap deflection angle (deg)
    "lift_curve_slope_flap_2d": 6.28, # 2D flap lift slope (per rad or deg, here per rad)
}

results = calculate_delta_cl_3d_wing(**inputs)

print("Results:")
for key, value in results.items():
    print(f"{key}: {value:.4f}")


Results:
CL_new: 2.3355
Cla (3D lift-curve slope per rad): 5.2429
Ka (Aerodynamic induction factor): 0.8344
Kb (Flap span effectiveness factor): 0.7200
Kc (Flap chord effectiveness factor): 1.0786
Kf (3D flap effectiveness factor): 0.6480
Delta_cla_section (Section lift increment): 2.1921
baseline_CL: 0.9151
